In [ ]:
#!/bin/bash
!kaggle datasets download msambare/fer2013

Dataset URL: https://www.kaggle.com/datasets/msambare/fer2013
License(s): DbCL-1.0
100% 60.3M/60.3M [00:01<00:00, 62.7MB/s]



In [ ]:
import zipfile
zip_path='/content/fer2013.zip'
extract_path='/content/fer2013'
with zipfile.ZipFile(zip_path,'r') as zip_ref:
  zip_ref.extractall(extract_path)

In [ ]:
import torch
import torchvision

In [ ]:
from torchvision import transforms,models
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(5),
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.2, 0.2),
        shear=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

valid_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
train_data=torchvision.datasets.ImageFolder(root='/content/fer2013/train',transform=train_transform)
test_data=torchvision.datasets.ImageFolder(root='/content/fer2013/test',transform=valid_transform)
#

In [ ]:
train_size = int(0.8 * len(train_data))
valid_size = len(train_data) - train_size

train_dataset, valid_dataset = torch.utils.data.random_split(
    train_data,
    [train_size, valid_size]
)

valid_dataset.dataset.transform = valid_transform

In [ ]:
from torch.utils.data import DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_data,
    batch_size=64,
    shuffle=False,
    num_workers=2
)

In [ ]:
images,labels=next(iter(train_loader))
images.shape

torch.Size([64, 3, 224, 224])

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

class EmotionModel(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()

        # Pretrained ResNet50
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

        # Use ResNet50 only as feature extractor
        self.base_model = nn.Sequential(
            *list(resnet.children())[:-2]
        )

        # Freeze ResNet50
        for param in self.base_model.parameters():
            param.requires_grad = False

        # Custom CNN layers
        self.conv1 = nn.Conv2d(
            2048, 128, kernel_size=5, padding=2
        )
        self.bn1 = nn.BatchNorm2d(128)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(0.25)

        self.conv2 = nn.Conv2d(
            128, 512, kernel_size=3, padding=1
        )
        self.bn2 = nn.BatchNorm2d(512)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(0.25)

        self.conv3 = nn.Conv2d(
            512, 512, kernel_size=3, padding=1
        )
        self.bn3 = nn.BatchNorm2d(512)
        self.relu3 = nn.ReLU()
        self.dropout3 = nn.Dropout(0.25)

        # Instead of Flattening the entire feature map,
        # use Adaptive Average Pooling
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        # Fully connected layers
        self.fc1 = nn.Linear(512, 256)
        self.bn4 = nn.BatchNorm1d(256)
        self.relu4 = nn.ReLU()
        self.dropout4 = nn.Dropout(0.25)

        self.fc2 = nn.Linear(256, 512)
        self.bn5 = nn.BatchNorm1d(512)
        self.relu5 = nn.ReLU()
        self.dropout5 = nn.Dropout(0.25)

        self.fc3 = nn.Linear(512, num_classes)

    def forward(self, x):

        # ResNet50 feature extraction
        x = self.base_model(x)

        # Custom CNN
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.dropout1(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.dropout2(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = self.relu3(x)
        x = self.dropout3(x)

        # Global pooling
        x = self.global_pool(x)

        x = torch.flatten(x, 1)

        # Fully connected
        x = self.fc1(x)
        x = self.bn4(x)
        x = self.relu4(x)
        x = self.dropout4(x)

        x = self.fc2(x)
        x = self.bn5(x)
        x = self.relu5(x)
        x = self.dropout5(x)

        x = self.fc3(x)

        return x

In [ ]:
model=EmotionModel()
device=torch.device(
    'cuda' if torch.cuda.is_available()
    else 'cpu'
)
model = model.to(device)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 131MB/s]


In [ ]:
device

device(type='cuda')

In [ ]:
import torch.nn as nn
criterion=nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.001
)


In [ ]:
num_epochs = 5
best_val_acc = 0.0

for epoch in range(num_epochs):

    # =========================
    # Training
    # =========================
    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_loss = train_loss / len(train_loader)
    train_acc = 100 * train_correct / train_total

    # =========================
    # Validation
    # =========================
    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_loss = val_loss / len(valid_loader)
    val_acc = 100 * val_correct / val_total

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc:.2f}% "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_acc:.2f}%"
    )

    # =========================
    # Save best model
    # =========================
    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(
            model.state_dict(),
            "best_resnet50_emotion.pth"
        )

        print("Best model saved!")

Epoch [1/5] Train Loss: 1.5831 Train Acc: 38.52% Val Loss: 1.4580 Val Acc: 43.99%
Best model saved!
Epoch [2/5] Train Loss: 1.3684 Train Acc: 48.16% Val Loss: 1.4275 Val Acc: 46.60%
Best model saved!
Epoch [3/5] Train Loss: 1.2139 Train Acc: 55.19% Val Loss: 1.4047 Val Acc: 47.67%
Best model saved!
Epoch [4/5] Train Loss: 1.0324 Train Acc: 62.15% Val Loss: 1.4443 Val Acc: 47.41%
Epoch [5/5] Train Loss: 0.8261 Train Acc: 70.26% Val Loss: 1.5371 Val Acc: 46.67%


In [ ]:
correct=0
total=0
model.eval()
with torch.no_grad():
  for images,labels in test_loader:
    images=images.to(device)
    labels=labels.to(device)
    outputs=model(images)
    _, predicted = torch.max(outputs, 1)
    total += labels./
    size(0)
    correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f"Accuracy: {accuracy:.2f}%")


Accuracy: 30.25%
